# 03 — ExGAN for tabular PLOS data

Port of **ExGAN** (Bhatia, Jain & Hooi, AAAI 2021; code: github.com/Stream-AD/ExGAN) from rainfall images to tabular clinical data.

| ExGAN (paper) | This notebook |
|---|---|
| Image, DCGAN | Encounter row, MLP generator + critic (WGAN-GP) |
| Extremeness E(x) = total rainfall | E = LOS (days) |
| Distribution shifting: keep top *c* fraction, refill with GAN samples, warm-start retrain, *k* times | Same. c = 0.75, k chosen so c^k ≈ PLOS share (lands on the PLOS region) |
| EVT: fit GPD to E, condition generator on e ~ GPD | Same. GPD fitted to real train PLOS LOS above the PLOS threshold |
| Extremeness loss \|e − E(G(z,e))\| | E is a column here, so the generated row's LOS is **set to e**; the critic judges whether the other features fit that LOS. The extremeness loss is identically 0 |

Two variants are trained with **equal compute**, as an ablation:
- **ExGAN**: distribution shifting + EVT conditioning
- **ExGAN-noshift**: conditional GAN on all train rows + EVT conditioning (no shifting)

Both are scored with the same fidelity table as `02`, next to SMOTE-NC, GaussianCopula, CTGAN, TVAE.
**Targets from 02:** mode error ≤ 0.06, extreme coverage ≈ 0.10 with LOS max ≥ 90, DCR ratio ≥ 1.


In [ ]:
# Cell 1 — paths and settings
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/genai_synthea/run_01"
OUT  = f"{BASE}/genai"
SYN_DIR = f"{OUT}/synthetic"
RES_DIR = f"{OUT}/results"

SEED   = 42
N_SYN  = 1000          # synthetic PLOS rows per variant
C      = 0.75          # ExGAN shift fraction (paper default)
INIT_EPOCHS  = 300     # first GAN
SHIFT_EPOCHS = 100     # warm-start epochs per shift
BATCH, Z_DIM, HIDDEN = 128, 64, 256
N_CRITIC, GP_W, LR = 5, 10.0, 2e-4
GUMBEL_TAU = 0.2


In [ ]:
# Cell 2 — load data
import os, json, time, math, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import genpareto
warnings.filterwarnings("ignore")
os.makedirs(SYN_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

meta  = json.load(open(f"{OUT}/meta.json"))
train = pd.read_parquet(f"{OUT}/train.parquet")
val   = pd.read_parquet(f"{OUT}/val.parquet")
THR, MODES = meta["plos_threshold"], meta["modes"]
CAT, BIN   = meta["cat_cols"], meta["bin_cols"]
NUM = meta["num_cols"] + ["target"]
GEN_COLS = NUM + CAT + BIN

DEV = "cuda" if torch.cuda.is_available() else "cpu"
real_tail = train[train["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
pos_share = len(real_tail) / len(train)
K = int(math.floor(math.log(pos_share) / math.log(C)))
print(f"Device: {DEV} | train rows {len(train)} | PLOS {len(real_tail)} ({pos_share:.1%})")
print(f"Shifts k = {K}  -> c^k = {C**K:.3f} of the data kept (PLOS share {pos_share:.3f})")


In [ ]:
# Cell 3 — encoder / decoder (age z-score, one-hot categoricals, 2-way one-hot per condition flag)
class Codec:
    def __init__(self, df):
        self.age_mu, self.age_sd = df["anchor_age"].mean(), df["anchor_age"].std()
        self.age_min, self.age_max = df["anchor_age"].min(), df["anchor_age"].max()
        logl = np.log(df["target"].clip(lower=1))
        self.l_mu, self.l_sd = logl.mean(), logl.std()
        self.levels = {c: sorted(df[c].astype(str).unique()) for c in CAT}
        self.cat_sizes = [len(self.levels[c]) for c in CAT]
        self.n_bin = len(BIN)
        self.dim = 1 + sum(self.cat_sizes) + 2 * self.n_bin

    def encode_x(self, df):
        parts = [((df["anchor_age"].values - self.age_mu) / self.age_sd)[:, None]]
        for c in CAT:
            v = df[c].astype(str).values
            parts.append(np.stack([(v == l) for l in self.levels[c]], 1).astype(np.float32))
        b = df[BIN].values.astype(np.float32)
        parts.append(np.stack([1 - b, b], 2).reshape(len(df), -1))
        return np.concatenate(parts, 1).astype(np.float32)

    def encode_e(self, los):
        return ((np.log(np.clip(los, 1, None)) - self.l_mu) / self.l_sd).astype(np.float32)[:, None]

    def decode_e(self, e):
        return np.exp(e.ravel() * self.l_sd + self.l_mu)

    def decode(self, x, los):
        out, i = {}, 1
        out["anchor_age"] = np.clip(np.round(x[:, 0] * self.age_sd + self.age_mu), self.age_min, self.age_max)
        out["target"] = np.round(los)
        for c, s in zip(CAT, self.cat_sizes):
            out[c] = np.array(self.levels[c])[x[:, i:i + s].argmax(1)]; i += s
        flags = x[:, i:].reshape(len(x), self.n_bin, 2).argmax(2)
        df = pd.DataFrame(out)
        df = pd.concat([df, pd.DataFrame(flags, columns=BIN)], axis=1)
        return df[GEN_COLS]

codec = Codec(train)
print("Encoded row dim:", codec.dim)


In [ ]:
# Cell 4 — networks + WGAN-GP training loop
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(Z_DIM + 1, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, codec.dim))
    def forward(self, z, e):
        h = self.net(torch.cat([z, e], 1))
        outs, i = [h[:, :1]], 1
        for s in codec.cat_sizes:
            outs.append(F.gumbel_softmax(h[:, i:i + s], tau=GUMBEL_TAU)); i += s
        b = h[:, i:].reshape(-1, codec.n_bin, 2)
        outs.append(F.gumbel_softmax(b, tau=GUMBEL_TAU, dim=2).reshape(len(h), -1))
        return torch.cat(outs, 1)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(codec.dim + 1, HIDDEN), nn.LeakyReLU(0.2),
                                 nn.Linear(HIDDEN, HIDDEN), nn.LeakyReLU(0.2),
                                 nn.Linear(HIDDEN, 1))
    def forward(self, x, e):
        return self.net(torch.cat([x, e], 1))

def grad_penalty(D, real, fake, e):
    a = torch.rand(len(real), 1, device=DEV)
    mix = (a * real + (1 - a) * fake).requires_grad_(True)
    g = torch.autograd.grad(D(mix, e).sum(), mix, create_graph=True)[0]
    return ((g.norm(2, dim=1) - 1) ** 2).mean()

def train_gan(G, D, X, E, epochs, optG, optD):
    X, E = torch.tensor(X, device=DEV), torch.tensor(E, device=DEV)
    n = len(X)
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEV)
        for b in range(0, n, BATCH):
            idx = perm[b:b + BATCH]
            real, e = X[idx], E[idx]
            for _ in range(N_CRITIC):
                fake = G(torch.randn(len(idx), Z_DIM, device=DEV), e).detach()
                lossD = D(fake, e).mean() - D(real, e).mean() + GP_W * grad_penalty(D, real, fake, e)
                optD.zero_grad(); lossD.backward(); optD.step()
            lossG = -D(G(torch.randn(len(idx), Z_DIM, device=DEV), e), e).mean()
            optG.zero_grad(); lossG.backward(); optG.step()
    return float(lossD), float(lossG)

@torch.no_grad()
def generate(G, los):
    G.eval()
    e = torch.tensor(codec.encode_e(np.asarray(los, dtype=float)), device=DEV)
    x = G(torch.randn(len(e), Z_DIM, device=DEV), e).cpu().numpy()
    G.train()
    return x

def new_models():
    G, D = Generator().to(DEV), Critic().to(DEV)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9))
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.9))
    return G, D, optG, optD


In [ ]:
# Cell 5 — EVT: fit a Generalized Pareto Distribution to real PLOS LOS above the threshold
exceed = real_tail["target"].values - THR
xi, loc, sigma = genpareto.fit(exceed, floc=0)
LOS_CAP = 1.2 * real_tail["target"].max()      # guard against unbounded draws when xi > 0
print(f"GPD fit on {len(exceed)} exceedances: shape xi = {xi:.3f}, scale sigma = {sigma:.2f}")
print("xi > 0 means a heavy tail (no upper bound); xi < 0 means a bounded tail")

def sample_los_gpd(n, rng):
    los = THR + genpareto.rvs(xi, loc=0, scale=sigma, size=n, random_state=rng)
    return np.clip(np.maximum(np.round(los), math.floor(THR) + 1), None, LOS_CAP)

def los_at_tail_prob(tau):
    """ExGAN-style: LOS whose exceedance probability among PLOS stays is tau (inverse GPD CDF)."""
    return THR + genpareto.ppf(1 - tau, xi, loc=0, scale=sigma)

for tau in [0.10, 0.05, 0.01]:
    print(f"  1-in-{int(1/tau)} PLOS stay: LOS ~ {los_at_tail_prob(tau):.0f} days")


In [ ]:
# Cell 6 — ExGAN: distribution shifting (Algorithm 1) + EVT-conditioned sampling (Algorithm 2)
def run_exgan(shift=True, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    X = codec.encode_x(train[GEN_COLS]); los = train["target"].values.astype(float)
    N = len(X)
    G, D, optG, optD = new_models()
    t0 = time.time()
    train_gan(G, D, X, codec.encode_e(los), INIT_EPOCHS, optG, optD)
    if shift:
        is_real = np.ones(N, bool)
        for i in range(1, K + 1):
            cut = np.quantile(los, 1 - C)                       # keep the top c fraction
            keep = los >= cut
            X, los, is_real = X[keep], los[keep], is_real[keep]
            n_new = N - len(X)                                   # refill with GAN samples above the cut
            e_new = rng.choice(los, size=n_new, replace=True)
            X = np.concatenate([X, generate(G, e_new)]); los = np.concatenate([los, e_new])
            is_real = np.concatenate([is_real, np.zeros(n_new, bool)])
            train_gan(G, D, X, codec.encode_e(los), SHIFT_EPOCHS, optG, optD)   # warm start
            print(f"  shift {i}/{K}: cut LOS >= {cut:.1f} d | real rows kept {is_real.sum():4d} "
                  f"({is_real.mean():.0%} of set) | {time.time()-t0:.0f}s")
    else:
        train_gan(G, D, X, codec.encode_e(los), K * SHIFT_EPOCHS, optG, optD)  # equal compute
    e = sample_los_gpd(N_SYN, rng)
    syn = codec.decode(generate(G, e), e)
    return syn, G, time.time() - t0

results_syn, fit_time, models = {}, {}, {}
for name, shift in [("ExGAN", True), ("ExGAN-noshift", False)]:
    print(f"Training {name} ...")
    syn, G, t = run_exgan(shift=shift)
    results_syn[name], fit_time[name], models[name] = syn, t, G
    syn.to_parquet(f"{SYN_DIR}/{name}.parquet", index=False)
    print(f"{name}: {len(syn)} rows in {t:.0f}s")


In [ ]:
# Cell 7 — load 02 baselines and score all generators with the same fidelity metrics
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.neighbors import NearestNeighbors

synthetic = {}
for name in ["SMOTE-NC", "GaussianCopula", "CTGAN", "TVAE"]:
    p = f"{SYN_DIR}/{name}.parquet"
    if os.path.exists(p): synthetic[name] = pd.read_parquet(p)
synthetic.update(results_syn)
fit_time_02 = {}
if os.path.exists(f"{RES_DIR}/02_fidelity.csv"):
    fit_time_02 = pd.read_csv(f"{RES_DIR}/02_fidelity.csv", index_col=0)["fit_s"].dropna().to_dict()

def mode_flags(d):
    f = pd.DataFrame({k: d[[c for c in v if c in d.columns]].max(axis=1) for k, v in MODES.items()}, index=d.index)
    f["unexplained"] = (f.sum(axis=1) == 0).astype(int)
    return f

real_flags = mode_flags(real_tail).mean()
real_p90 = real_tail["target"].quantile(0.90)
age_mu, age_sd = real_tail["anchor_age"].mean(), real_tail["anchor_age"].std()
los_mu, los_sd = real_tail["target"].mean(), real_tail["target"].std()
cat_levels = {c: sorted(real_tail[c].astype(str).unique()) for c in CAT}
def encode(d):
    parts = [((d["anchor_age"] - age_mu) / age_sd).to_frame(), ((d["target"] - los_mu) / los_sd).to_frame(), d[BIN].astype(float)]
    for c in CAT:
        parts.append(pd.DataFrame({f"{c}={l}": (d[c].astype(str) == l).astype(float) for l in cat_levels[c]}, index=d.index))
    return pd.concat(parts, axis=1).values

nn_ = NearestNeighbors(n_neighbors=1).fit(encode(real_tail))
holdout = val[val["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
dcr_holdout = np.median(nn_.kneighbors(encode(holdout))[0])
real_corr = real_tail[["anchor_age", "target"] + BIN].corr().fillna(0).values

rows = []
for name, s in synthetic.items():
    s = s[GEN_COLS].copy()
    for c in BIN: s[c] = s[c].astype(int)
    f = mode_flags(s).mean()
    d_syn = nn_.kneighbors(encode(s))[0].ravel()
    cd = np.abs(s[["anchor_age", "target"] + BIN].corr().fillna(0).values - real_corr)
    rows.append({"generator": name, "validity": float((s["target"] > THR).mean()),
        "LOS_wasserstein": wasserstein_distance(real_tail["target"], s["target"]),
        "LOS_KS": ks_2samp(real_tail["target"], s["target"]).statistic,
        "extreme_cov": (s["target"] > real_p90).mean(), "LOS_max": s["target"].max(),
        **{f"mode_{k}": f[k] for k in f.index},
        "mode_abs_err": (f - real_flags).abs().mean(),
        "cond_prev_err": (s[BIN].mean() - real_tail[BIN].mean()).abs().mean(),
        "corr_err": cd[np.triu_indices_from(cd, 1)].mean(),
        "exact_copies": (d_syn < 1e-6).mean(), "DCR_ratio": np.median(d_syn) / dcr_holdout,
        "fit_s": fit_time.get(name, fit_time_02.get(name, np.nan))})

real_row = {"generator": "REAL (train PLOS)", "validity": 1.0, "LOS_wasserstein": 0, "LOS_KS": 0,
            "extreme_cov": (real_tail["target"] > real_p90).mean(), "LOS_max": real_tail["target"].max(),
            **{f"mode_{k}": real_flags[k] for k in real_flags.index}}
fid = pd.DataFrame([real_row] + rows).set_index("generator")
fid.to_csv(f"{RES_DIR}/03_fidelity_all.csv")
print(f"Real PLOS rows {len(real_tail)} | LOS p90 in tail {real_p90:.0f} d | holdout median DCR {dcr_holdout:.2f}")
print("Targets: mode_abs_err <= 0.06 | extreme_cov ~ 0.10 | LOS_max >= 90 | DCR_ratio >= 1")
fid.round(3).T


In [ ]:
# Cell 8 — scorecard against the 02 targets
score = pd.DataFrame({
    "modes_ok (err<=0.06)":   fid["mode_abs_err"] <= 0.06,
    "extremes_ok (cov>=0.08)": fid["extreme_cov"] >= 0.08,
    "max_ok (>=90 d)":         fid["LOS_max"] >= 90,
    "no_copying (DCR>=1)":     fid["DCR_ratio"] >= 1.0,
}).drop(index="REAL (train PLOS)")
score["targets_met"] = score.sum(axis=1)
print(score.sort_values("targets_met", ascending=False).to_string())


In [ ]:
# Cell 9 — plots: LOS tail and mode coverage, all generators
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
hi = max(real_tail["target"].max(), max(s["target"].max() for s in synthetic.values()))
bins = np.arange(THR, hi + 3, 3)
axes[0].hist(real_tail["target"], bins=bins, density=True, histtype="step", lw=2.8, color="black", label="REAL")
for name, s in synthetic.items():
    lw = 2.2 if name.startswith("ExGAN") else 1.0
    axes[0].hist(s["target"], bins=bins, density=True, histtype="step", lw=lw, label=name)
axes[0].set_yscale("log"); axes[0].set_xlabel("LOS (days)"); axes[0].set_title("PLOS tail: LOS"); axes[0].legend(fontsize=8)
mc = [c for c in fid.columns if c.startswith("mode_") and c != "mode_abs_err"]
fid[mc].rename(columns=lambda c: c.replace("mode_", "")).T.plot.bar(ax=axes[1], rot=0)
axes[1].set_ylabel("share of PLOS rows"); axes[1].set_title("Mode coverage"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{RES_DIR}/03_plots.png", dpi=150); plt.show()


In [ ]:
# Cell 10 — ExGAN's own feature: generate patients at a chosen extremeness level
for tau in [0.10, 0.01]:
    L = los_at_tail_prob(tau)
    rows_ = codec.decode(generate(models["ExGAN"], np.full(200, L)), np.full(200, round(L)))
    f = mode_flags(rows_).mean().round(2).to_dict()
    print(f"1-in-{int(1/tau)} PLOS stay (LOS ~ {L:.0f} d): mean age {rows_['anchor_age'].mean():.0f} | modes {f}")
